# 14 — X-IIoTID Core19 | Laptop Resource & Inference Benchmark

**J26-CS-336 · Network Flow V2 · Gateway deployment readiness**

**Purpose:** Evaluate the **five already-trained, full-training** model artifacts for storage size, runtime RAM and CPU usage, model-loading time, single-flow inference latency, and batch throughput. Create CSV/JSON evidence and report-quality PNG plots.

**What this notebook does NOT do:** No retraining; **never opens the test split**; does not measure Raspberry Pi performance; does not measure live packet capture, flow generation, or feature-extraction time. The input is a deterministic sample of **frozen validation data**, cleaned using the **existing `x_iiotid_model_runner.clean_features` function**, and fed to the *saved full preprocessing + classification pipeline*.

**Interpretation:** A lightweight *model file* is not automatically a lightweight *detection system*. CPU and RAM here are measured on this Windows laptop. Re-run comparable benchmarks on the actual Raspberry Pi later.

**Prerequisites:** Notebooks 08–12 trained and saved `.joblib` pipelines; matching validation CSV metrics; active `.venv` kernel; Python packages `joblib`, `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `lightgbm`, `psutil` installed. To install just the extra monitoring dependency in the notebook kernel, run `%pip install psutil` **manually** in a separate cell if needed.

In [ ]:
from pathlib import Path
import sys, os, json, importlib.util
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
assert ROOT.is_dir(), f"Research folder missing: {ROOT}"
assert (ROOT / "src" / "x_iiotid_model_runner.py").is_file(), "Missing shared model runner"

AUDIT_DIR = ROOT / "reports" / "audits"
FIG_DIR = ROOT / "reports" / "figures"
MODEL_DIR = ROOT / "models" / "x_iiotid"
VALIDATION = ROOT / "data" / "splits" / "x_iiotid_core19_validation_v1.csv.gz"
for folder in [AUDIT_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
assert VALIDATION.is_file(), f"Frozen validation split missing: {VALIDATION}"

MODELS = ["decision_tree", "random_forest", "extra_trees", "lightgbm", "adaboost"]
DISPLAY_NAMES = {"decision_tree":"Decision Tree", "random_forest":"Random Forest",
                 "extra_trees":"Extra Trees", "lightgbm":"LightGBM", "adaboost":"AdaBoost"}

missing_packages = [name for name in ["psutil", "joblib", "sklearn", "numpy", "pandas", "matplotlib", "lightgbm"]
                    if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f"Install these packages in the active notebook kernel, then rerun: {missing_packages}. Example: %pip install psutil")

print("Active Python:", sys.executable)
print("Workspace:", ROOT)
print("Host:", sys.platform, "| Logical CPUs:", os.cpu_count())
print("Input partition: VALIDATION only (held-out test is sealed)")

## A. Verify the five full-trained reports & model files

The report row count verifies what the evaluation **recorded**. The saved `.joblib` format does **not** embed an independently verifiable training-row count, so we also check that each model artifact exists and whether its file modification time is consistent with the report. Matching times are **supporting evidence, not cryptographic provenance**.

In [ ]:
rows = []
manifest_hashes = set()
for model in MODELS:
    model_file = MODEL_DIR / f"{model}_core19_v1.joblib"
    metric_file = AUDIT_DIR / f"x_iiotid_{model}_validation_v1.csv"
    if not model_file.is_file():
        raise FileNotFoundError(f"Missing trained pipeline: {model_file}")
    if not metric_file.is_file():
        raise FileNotFoundError(f"Missing validation report: {metric_file}")
    rec = pd.read_csv(metric_file).iloc[0]
    if int(rec["train_rows_used"]) != 574_585 or int(rec["validation_rows"]) != 123_125:
        raise AssertionError(f"{model}: report does not show full training + expected validation rows")
    if str(rec["test_evaluated"]).strip().lower() not in ("false", "0"):
        raise AssertionError(f"{model}: report says test was evaluated; stop and investigate")
    manifest_hashes.add(str(rec["frozen_manifest_sha256"]))
    rows.append({
        "model": model,
        "train_rows_reported": int(rec["train_rows_used"]),
        "validation_rows": int(rec["validation_rows"]),
        "f1_attack": float(rec["f1_attack"]),
        "false_positive_rate": float(rec["false_positive_rate"]),
        "model_size_mib": round(model_file.stat().st_size / (1024**2), 3),
        "model_modified_utc": pd.Timestamp(model_file.stat().st_mtime, unit="s", tz="UTC").isoformat(),
        "metrics_modified_utc": pd.Timestamp(metric_file.stat().st_mtime, unit="s", tz="UTC").isoformat(),
        "model_not_older_than_metrics": model_file.stat().st_mtime >= metric_file.stat().st_mtime,
    })
assert len(manifest_hashes) == 1, "Different frozen manifest hashes! Do not compare models yet."
metadata_df = pd.DataFrame(rows)
display(metadata_df)
if not metadata_df["model_not_older_than_metrics"].all():
    print("WARNING: Some saved model files predate their reported metrics; verify which training run created them.")
print("Same frozen manifest hash:", next(iter(manifest_hashes)))

## B. Fair execution method — fresh Python process per model

A separate Python process is launched for **each saved model**, so models do not share memory and influence one another. The worker imports the **exact cleaning function used in training**, samples only the validation split deterministically, checks that the model is an sklearn `Pipeline` with `preprocessing` and `classifier`, and measures the complete pipeline's `.predict()`.

- **RSS** (resident set size): actual memory occupied by the Python process. `after-load RSS − baseline RSS` is an *approximate model incremental memory cost*, not a perfect model-only allocation measure.
- **Peak RSS:** largest observed process memory during the prediction run. Sampling can miss extremely short spikes.
- **Latency:** wall time per prediction call. For single-flow requests, use **median (p50)** and **95th percentile (p95)**; for the 1,000-flow batch, compute flows/second.
- **CPU:** process CPU time relative to wall time; **100% = one fully used logical CPU**, so multi-threaded models can exceed 100%. It is **not** whole-machine CPU utilization.
- **Inputs:** 1,024 deterministically sampled *original validation rows*; `clean_features()` does numeric conversion and placeholder processing before timing, because this function is currently outside the saved pipeline. This benchmark includes **saved pipeline preprocessing** but excludes the separate `clean_features()` step, network capture and live feature computation.

**Important:** Process CPU measurements and RSS on your laptop do not substitute for a real Pi benchmark. Small timings can be noisy; repeated trials and identical hardware are needed for defensible comparisons.

In [ ]:
WORKER_CODE = r'''
import sys, os, json, time, threading, gc
from pathlib import Path

root = Path(sys.argv[1]); model_name = sys.argv[2]
sys.path.insert(0, str(root / "src"))
import numpy as np
import pandas as pd
import joblib
import psutil
from sklearn.pipeline import Pipeline
from x_iiotid_model_runner import CORE19, clean_features

src = root / "data/splits/x_iiotid_core19_validation_v1.csv.gz"
model_file = root / "models/x_iiotid" / (model_name + "_core19_v1.joblib")
assert src.is_file() and model_file.is_file()

# Scan the VALIDATION partition only; obtain a deterministic spread of rows.
sampled = []
for ci, chunk in enumerate(pd.read_csv(src, dtype=str, keep_default_na=False,
                                       usecols=CORE19, chunksize=10000)):
    sampled.append(chunk.sample(n=min(128, len(chunk)), random_state=42 + ci))
raw = pd.concat(sampled, ignore_index=True).sample(n=1024, random_state=42)
X = clean_features(raw).reset_index(drop=True)
del sampled, chunk, raw
gc.collect()

proc = psutil.Process(os.getpid())
def rss(): return proc.memory_info().rss
base_rss = rss()
start_load = time.perf_counter()
model = joblib.load(model_file)  # Load only trusted, locally trained artifacts.
load_sec = time.perf_counter() - start_load
post_load_rss = rss()
if not isinstance(model, Pipeline) or not {'preprocessing','classifier'}.issubset(model.named_steps):
    raise TypeError(f"Saved {model_name} artifact is not the expected preprocessing+classifier Pipeline")

batch_sizes = (1, 32, 256, 1000)
batches = {size: X.iloc[:size].copy() for size in batch_sizes}
peak = [rss()]
stop = threading.Event()
def poll():
    while not stop.wait(0.005):
        peak[0] = max(peak[0], rss())
thread = threading.Thread(target=poll, daemon=True)
thread.start()

result = {}
try:
    for batch_size in batch_sizes:
        batch = batches[batch_size]
        for _ in range(3):
            out = model.predict(batch)
            assert len(out) == batch_size
        loops = 60 if batch_size == 1 else (25 if batch_size == 32 else 12)
        times_ms = []
        for _ in range(loops):
            t0 = time.perf_counter()
            model.predict(batch)
            times_ms.append(1000 * (time.perf_counter() - t0))
        arr = np.array(times_ms)
        result[f'batch{batch_size}_p50_ms'] = float(np.median(arr))
        result[f'batch{batch_size}_p95_ms'] = float(np.percentile(arr, 95))
        if batch_size == 1000:
            result['batch1000_throughput_flows_s'] = float(1000 / (np.mean(arr) / 1000))

    # Sustained inference CPU measurement, at least 1 second or 200 repeats.
    batch = batches[1000]
    t0 = time.perf_counter()
    c0 = proc.cpu_times()
    loops = 0
    while loops < 200 and (loops < 2 or time.perf_counter() - t0 < 1.0):
        model.predict(batch)
        loops += 1
    wall = time.perf_counter() - t0
    c1 = proc.cpu_times()
    process_cpu_seconds = (c1.user + c1.system) - (c0.user + c0.system)
    result['cpu_pct_one_logical_core'] = float(100 * process_cpu_seconds / wall)
    result['cpu_measurement_wall_seconds'] = float(wall)
    result['cpu_measurement_prediction_loops'] = loops
finally:
    peak[0] = max(peak[0], rss())
    stop.set()
    thread.join(timeout=1)

result.update({
    'model': model_name,
    'input_validation_rows_sampled': int(len(X)),
    'model_load_seconds': float(load_sec),
    'rss_baseline_mib': float(base_rss / 2**20),
    'rss_after_load_mib': float(post_load_rss / 2**20),
    'model_rss_increment_mib': float((post_load_rss - base_rss) / 2**20),
    'peak_process_rss_mib': float(peak[0] / 2**20),
    'cpu_logical_cores': os.cpu_count(),
    'python_version': sys.version.split()[0],
    'os_platform': sys.platform,
    'preprocessing_in_predict': True,
    'clean_features_timed': False,
    'network_capture_timed': False,
    'test_evaluated': False,
})
print('BENCHMARK_RESULT_JSON=' + json.dumps(result, allow_nan=False))
'''
print("Benchmark worker prepared; separate process per model.")

## C. Run all five saved models — **no training**

One subprocess per model using the **same active notebook kernel interpreter**. Errors are shown without hiding their full output. Expect several seconds to a minute depending on the models and laptop performance. `subprocess` is necessary for isolation; avoid running these measurements concurrently with other heavy applications.

In [ ]:
import subprocess, time

benchmark_rows = []
failures = {}
for model_name in MODELS:
    print(f"Benchmarking saved {DISPLAY_NAMES[model_name]} pipeline ...", flush=True)
    started = time.perf_counter()
    try:
        completed = subprocess.run(
            [sys.executable, "-c", WORKER_CODE, str(ROOT), model_name],
            text=True, capture_output=True, timeout=240, check=True,
            cwd=str(ROOT)
        )
        output = [line.split("=", 1)[1] for line in completed.stdout.splitlines()
                  if line.startswith("BENCHMARK_RESULT_JSON=")]
        if not output:
            raise RuntimeError("Worker returned no results. stdout:\n" + completed.stdout
                               + "\nstderr:\n" + completed.stderr)
        benchmark_rows.append(json.loads(output[-1]))
        print(f"  Success in {time.perf_counter()-started:.1f}s")
    except Exception as exc:
        message = str(exc)
        if isinstance(exc, subprocess.CalledProcessError):
            message += "\nstdout:\n" + (exc.stdout or "") + "\nstderr:\n" + (exc.stderr or "")
        failures[model_name] = message
        print(f"  FAILED: {message[:1500]}")

if failures:
    print("\nFix benchmark errors before interpreting partial results:")
    for name, error in failures.items():
        print("\n", name, error[:3000])
    raise RuntimeError(f"{len(failures)} model(s) failed resource benchmarking")

resource_df = metadata_df.merge(pd.DataFrame(benchmark_rows), on="model", validate="one_to_one")
assert len(resource_df) == 5
view_columns = ["model", "model_size_mib", "model_load_seconds", "model_rss_increment_mib",
                "peak_process_rss_mib", "cpu_pct_one_logical_core", "batch1_p50_ms",
                "batch1_p95_ms", "batch1000_throughput_flows_s", "f1_attack"]
display(resource_df[view_columns].round(3))

## D. Save the raw comparison evidence

CSV and JSON include measurement caveats. Keep full precision in files, rounding only for display. Results are **laptop measurements** of **pre-cleaned validation inputs through the saved preprocessing-and-classifier pipeline**.

In [ ]:
from datetime import datetime, timezone

csv_out = AUDIT_DIR / "x_iiotid_five_model_laptop_resource_benchmark_v1.csv"
json_out = AUDIT_DIR / "x_iiotid_five_model_laptop_resource_benchmark_v1.json"
resource_df.to_csv(csv_out, index=False)
payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "Frozen X-IIoTID Core19 saved model laptop resource benchmark",
    "train_rows_reported_per_model": 574585,
    "validation_rows_available": 123125,
    "validation_rows_sampled_per_worker": 1024,
    "frozen_manifest_sha256": next(iter(manifest_hashes)),
    "hardware": "current laptop, not Raspberry Pi",
    "validation_sampling": "128 deterministic rows sampled from each 10k-row validation chunk; 1024 drawn from pooled sample",
    "model_loading": "only locally trained trusted joblib pipeline artifacts",
    "timed_prediction_includes": "fitted Pipeline.preprocessing and Pipeline.classifier",
    "timed_prediction_excludes": ["read_csv", "clean_features", "network packet capture", "flow aggregation", "dashboard/alerting"],
    "ram_definition": "process RSS; after-load increment includes allocator effects, peak RSS is sampled at 5ms intervals",
    "cpu_definition": "process CPU time / wall time; 100% is one logical CPU, >100% possible",
    "test_set_evaluated": False,
    "cross_hardware_comparison": "Do not directly compare laptop benchmark with Pi until tested on Pi",
    "results": resource_df.to_dict(orient="records"),
}
json_out.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved CSV:", csv_out)
print("Saved JSON:", json_out)

## E. Generate five report-ready PNG comparison charts

The charts deliberately avoid arbitrary scores and benchmark only measured quantities. Resource usage is a **trade-off**: smaller storage, lower memory, lower latency and good F1 are generally desirable, but throughput and actual gateway feature availability also matter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

order = ["decision_tree", "random_forest", "extra_trees", "lightgbm", "adaboost"]
plot_df = resource_df.set_index("model").loc[order]
labels = [DISPLAY_NAMES[m] for m in order]

plots = [
    ("model_size_mib", "Trained model file size", "Storage (MiB)", "x_iiotid_resource_model_file_size_v1.png"),
    ("peak_process_rss_mib", "Peak Python process RAM during inference", "Peak RSS (MiB)", "x_iiotid_resource_peak_process_ram_v1.png"),
    ("cpu_pct_one_logical_core", "Sustained inference process CPU", "CPU % (100%=one logical core)", "x_iiotid_resource_cpu_inference_v1.png"),
    ("batch1_p95_ms", "Single-flow inference p95 latency", "p95 wall time (ms / flow)", "x_iiotid_resource_single_flow_latency_v1.png"),
    ("batch1000_throughput_flows_s", "Pipeline batch inference throughput", "Flows / second (batch=1000)", "x_iiotid_resource_batch_throughput_v1.png"),
]

for field, title, ylabel, filename in plots:
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    values = plot_df[field].astype(float).values
    bars = ax.bar(labels, values)
    ax.set_title("X-IIoTID Core19 — " + title + " (laptop)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(values) * 1.18 if max(values) > 0 else 1)
    ax.grid(axis="y", alpha=0.25)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(rotation=10)
    fig.tight_layout()
    destination = FIG_DIR / filename
    fig.savefig(destination, dpi=250, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", destination)

## F. Interpret results for the Raspberry Pi deployment decision

1. **F1-score** indicates detection quality on a provisional **signature-disjoint** validation split; it is **not** independently confirmed real-world APT detection.
2. **Model file size** is storage on disk. It is not equal to peak RAM usage.
3. **Peak RSS** includes the Python interpreter, dependencies, sampled feature DataFrame and loaded model. Use this for sizing the full Pi detection service, rather than only model incremental RSS.
4. **Latency** here is per prepared flow's `Pipeline.predict()`, **excluding** packet capture, flow duration, window collection and `clean_features()` outside the saved pipeline. This is not end-to-end alert latency.
5. **CPU percentage** uses process time normalized to one logical CPU; a four-core Pi can have up to ~400% aggregate usage under this convention.
6. **Pi compatibility:** Train on the laptop; copy only the trusted `.joblib` model and matching dependency versions to the Pi. Since sklearn/lightgbm serialization is version-sensitive, verify the same package versions or re-export safely if needed.
7. **Final hardware evidence:** Repeat with Raspberry Pi 4/5 and its actual network gateway load and feature extractor; measure continuous monitoring, alert path, power/temperature/throttling and throughput under realistic traffic.

**Research caution:** `Service`, `Duration`, byte counts and placeholder patterns can create dataset-specific shortcuts. A high F1 or fast inference does not replace feature-semantic and dataset-generalization checks.

In [ ]:
# Easy-to-read deployment shortlist: order by validation F1 (quality), with resource costs alongside.
summary_cols = ["model", "f1_attack", "false_positive_rate", "model_size_mib",
                "peak_process_rss_mib", "batch1_p95_ms", "batch1000_throughput_flows_s",
                "cpu_pct_one_logical_core"]
shortlist = resource_df[summary_cols].sort_values("f1_attack", ascending=False).reset_index(drop=True)
display(shortlist.round(3))
print("Interpret jointly: F1 + false-positive rate + RAM + latency + gateway-extractable features.")
print("NO Raspberry Pi measurements were performed in this notebook.")

## G. Save notebook & commit this milestone (PowerShell terminal)

Once you have reviewed the results, save the notebook as:

`network_flow_v2/notebooks/x_iiotid/14_x_iiotid_resource_benchmark.ipynb`

Then run these commands in **PowerShell**, after verifying the branch and staged files. **Do not stage local models, raw data, validation splits, Python environments or credentials**.

```powershell
Set-Location 'D:\J26-CS-336-Network-Flow-V2'
git branch --show-current
# Must show feature/aazaf-network-flow-v2

git add -- network_flow_v2/notebooks/x_iiotid/14_x_iiotid_resource_benchmark.ipynb
git add -- network_flow_v2/reports/audits/x_iiotid_five_model_laptop_resource_benchmark_v1.csv
git add -- network_flow_v2/reports/audits/x_iiotid_five_model_laptop_resource_benchmark_v1.json
git add -- 'network_flow_v2/reports/figures/x_iiotid_resource_*.png'

git diff --cached --name-only
git diff --cached --check
# If expected files only:
git commit -m 'bench: profile five X-IIoTID classifiers for edge readiness'
git push
```

**Report these to your supervisor:** (a) five-model resource summary table, (b) model-size graph, (c) peak-process-RAM graph, (d) single-flow p95 latency, and (e) laptop-versus-Pi evaluation limitation.